In [1]:
# 1. QUAN TRỌNG: Quay về thư mục gốc /content trước để không bao giờ bị lỗi mất dấu thư mục
%cd /content

# 2. Cài công cụ hệ thống
!apt-get update -qq && apt-get install -y -qq ffmpeg zstd

# 3. Tải lại mã nguồn mới nhất
!rm -rf /content/Phat_trien_ung_dung_DT012
!git clone -b Tuan8 https://github.com/Minhthang1472/Phat_trien_ung_dung_DT012.git

# 4. Cài thư viện Python
%cd /content/Phat_trien_ung_dung_DT012/backend-ai
!pip install -q -r requirements.txt

# 5. TỰ ĐỘNG COPY KEY: Dù bạn thả file ở ngoài /content hay ở đâu nó cũng tự copy vào backend-ai!
import shutil, os
if os.path.exists('/content/serviceAccountKey.json'):
    shutil.copy('/content/serviceAccountKey.json', '/content/Phat_trien_ung_dung_DT012/backend-ai/serviceAccountKey.json')
    print("🔑 Đã tự động nạp khóa Firebase vào backend-ai!")

# 6. Tạo file .env tối ưu GPU
with open('/content/Phat_trien_ung_dung_DT012/backend-ai/.env', 'w') as f:
    f.write("""WHISPER_DEVICE=cuda
WHISPER_COMPUTE_TYPE=float16
WHISPER_MODEL_SIZE=base
OLLAMA_BASE_URL=http://localhost:11434
OLLAMA_MODEL=qwen2.5:3b
FIREBASE_CREDENTIALS_PATH=serviceAccountKey.json
""")

print("✅ Ô 1 XONG: Môi trường & Mã nguồn sẵn sàng!")


/content
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
Cloning into 'Phat_trien_ung_dung_DT012'...
remote: Enumerating objects: 542, done.
remote: Counting objects: 100% (542/542), done.
remote: Compressing objects: 100% (352/352), done.
remote: Total 542 (delta 316), reused 393 (delta 170), pack-reused 0 (from 0)
Receiving objects: 100% (542/542), 18.64 MiB | 21.55 MiB/s, done.
Resolving deltas: 100% (316/316), done.
/content/Phat_trien_ung_dung_DT012/backend-ai
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 1.3 MB

In [2]:
import subprocess
import time

# 1. Cài đặt zstd (công cụ giải nén mà Ollama yêu cầu)
print("📦 Đang cài đặt zstd...")
!apt-get update -qq && apt-get install -y -qq zstd

# 2. Cài đặt Ollama phiên bản Linux lên Colab
print("⏳ Đang cài đặt Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Bật Ollama Server chạy ngầm trong nền Colab
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)

# 4. Tải mô hình Qwen 2.5 (3B) về GPU Tesla T4
print("📥 Đang tải mô hình Qwen 2.5 về GPU Tesla T4 (khoảng 1.9GB)...")
!ollama pull qwen2.5:3b

print("\n✅ ĐÃ CÀI ĐẶT THÀNH CÔNG OLLAMA & QWEN 2.5 TRÊN GPU TESLA T4!")



📦 Đang cài đặt zstd...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
⏳ Đang cài đặt Ollama...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
📥 Đang tải mô hình Qwen 2.5 về GPU Tesla T4 (khoảng 1.9GB)...


✅ ĐÃ CÀI ĐẶT THÀNH CÔNG OLLAMA & QWEN 2.5 TRÊN GPU TESLA T4!


In [ ]:
!pip install "av<19"
!pip install -q nvidia-cublas-cu12 nvidia-cudnn-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 41.5 MB/s eta 0:00:00
  Attempting uninstall: av
    Found existing installation: av 19.0.2
    Uninstalling av-19.0.2:
      Successfully uninstalled av-19.0.2


In [ ]:
import os
import nvidia.cublas.lib
import nvidia.cudnn.lib
os.environ["LD_LIBRARY_PATH"] = f"{nvidia.cublas.lib.__path__[0]}:{nvidia.cudnn.lib.__path__[0]}:{os.environ.get('LD_LIBRARY_PATH', '')}"


In [6]:
import time, re, shutil, os

# 1. Tự copy key nếu để ở ngoài thư mục gốc /content
if os.path.exists('/content/serviceAccountKey.json'):
    shutil.copy('/content/serviceAccountKey.json', '/content/Phat_trien_ung_dung_DT012/backend-ai/serviceAccountKey.json')

# 2. Dọn dẹp tiến trình cũ
!pkill cloudflared
!pkill uvicorn

# 3. Tải và chạy Cloudflare Tunnel chế độ nohup chống tràn log
!wget -nc -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!nohup ./cloudflared tunnel --url http://127.0.0.1:8000 > tunnel.log 2>&1 &

print("⏳ Đang tạo đường hầm Cloudflare...")
time.sleep(8)

with open("tunnel.log", "r") as f:
    log_text = f.read()

urls = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_text)

if urls:
    print("\n" + "="*60)
    print(f"🎉 ĐƯỜNG LINK BACKEND CỦA BẠN LÀ:\n👉 {urls[-1]}")
    print("="*60)
    print(f"Kiểm tra tại: {urls[-1]}/docs\n")

    # 🔥 TỰ ĐỘNG ĐẨY LINK LÊN FIREBASE FIRESTORE
    try:
        import firebase_admin
        from firebase_admin import credentials, firestore
        if not firebase_admin._apps:
            cred = credentials.Certificate('/content/Phat_trien_ung_dung_DT012/backend-ai/serviceAccountKey.json')
            firebase_admin.initialize_app(cred)
        db = firestore.client()
        db.collection('system_config').document('server').set({
            'url': urls[-1],
            'updated_at': firestore.SERVER_TIMESTAMP
        })
        print("🔥 ĐÃ TỰ ĐỘNG LƯU LINK LÊN FIREBASE THÀNH CÔNG! (App sẽ tự nhận link)")
    except Exception as e:
        print(f"⚠️ Chưa ghi được lên Firebase: {e}")
else:
    print("⚠️ Log chi tiết:")
    print(log_text)

# 4. Bật Server FastAPI
%cd /content/Phat_trien_ung_dung_DT012/backend-ai
!uvicorn app.main:app --host 0.0.0.0 --port 8000



⏳ Đang tạo đường hầm Cloudflare...

🎉 ĐƯỜNG LINK BACKEND CỦA BẠN LÀ:
👉 https://eur-spread-priced-supplies.trycloudflare.com
Kiểm tra tại: https://eur-spread-priced-supplies.trycloudflare.com/docs

🔥 ĐÃ TỰ ĐỘNG LƯU LINK LÊN FIREBASE THÀNH CÔNG! (App sẽ tự nhận link)
/content/Phat_trien_ung_dung_DT012/backend-ai
INFO:     Started server process [18318]
INFO:     Waiting for application startup.
INFO:     Khởi tạo Firebase Admin SDK thành công!
INFO:     Kết nối Google Firebase Cloud Firestore THÀNH CÔNG! Sẵn sàng làm Kho Cache vĩnh cửu.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     42.119.202.124:0 - "OPTIONS /api/history?limit=25 HTTP/1.1" 200 OK
/usr/local/lib/python3.13/dist-packages/google/cloud/firestore_v1/base_collection.py:317: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)
INFO:     42.119.202

In [ ]:
%cd /content/Phat_trien_ung_dung_DT012
!git pull origin Tuan8


/content/Phat_trien_ung_dung_DT012
From https://github.com/Minhthang1472/Phat_trien_ung_dung_DT012
 * branch            Tuan8      -> FETCH_HEAD
Already up to date.
